In [55]:
!git clone https://github.com/openai/human-eval.git


fatal: destination path 'human-eval' already exists and is not an empty directory.


In [56]:
!pip install openai

In [57]:
import time
from dataclasses import dataclass


import os
from openai import OpenAI



class openai_Service():
    model_name = "gpt-3.5-turbo"
    save_history = True

    def __init__(self, save_history = True, model_name = "gpt-3.5-turbo") -> None:
        self.history = []
        self.save_history = save_history
        self.model_name = model_name
        self.api_key = "sk-you openai api key"
        pass

    def make_request(self, prompt):
        client = OpenAI(api_key=self.api_key)
        try:
            chat_completion = client.chat.completions.create(
                messages=[
                    {
                        "role": "user",
                        "content": prompt,
                    }
                ],
                model=self.model_name,
            )
            assistant_message = chat_completion.choices[0].message.content
            return assistant_message
        except Exception as e:
            print(f"An error occurred: {e}")


    def clear_history(self):
        self.history = []
        return 'clear_history success'

    def get_history(self):
        return self.history

    def get_name(self):
        return self.model_name

    pass

# if __name__ == '__main__':

#     service = openai_Service(model_name = "gpt-3.5-turbo")

#     initial_prompt = '''from typing import List
# def has_close_elements(numbers: List[float], threshold: float) -> bool:
#     """ Check if in given list of numbers, are any two numbers closer to each other than
#     given threshold.
#     >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
#     False
#     >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
#     True
#     """
# '''

#     print(service.make_request(initial_prompt))



In [91]:
import os
def set_cuda_visible_devices(cuda_devices):
    os.environ["CUDA_VISIBLE_DEVICES"] = cuda_devices


# from api.chatglm_pro_api import ChatGLMProService
# from api.chatglm_turbo_api import ChatGLM_TurboService
# from api.minimax_api import MinimaxService
# from api.baichuan2_7b_api import Baichuan2_7b_Service
# from api.qwen_7b_api import Qwen_7b_chat_Service

class PairDevFramework:
    def __init__(self, driver, navigator, iscodeLLM=False):
        self.iscodeLLM = iscodeLLM
        # Initialize state
        self.default_driver_prompt = "You are now the driver in pair programming and your task is to write code. Please follow the instructions below to generate the code, and only return the full code content, not the extra text."
        self.default_navigator_prompt = "You are now the Navigator in pair programming and your task is to review the code and provide feedback. Please review the code below to indicate if there is an error, or just return [NOERROR] if there is no error."
        self.code = ""  # Current generated code
        self.errors = []  # Errors in the code
        self.driver = driver  # Initialize the driver as LLM1
        self.navigator = navigator  # Initialize the navigator as LLM2
        self.question = ""

    def reset(self):
        self.driver.clear_history()
        self.navigator.clear_history()
        self.code = ""  # Current generated code
        self.errors = []  # Errors in the code
        self.question = ""

    def switch_roles(self):
        # Switch roles
        self.driver, self.navigator = self.navigator, self.driver
        self.driver.make_request(self.default_driver_prompt)
        self.navigator.make_request(self.default_navigator_prompt)

    def generate_code(self, prompt):
        self.code = prompt + self.driver.make_request(prompt)
        # self.code = self.driver.make_request(prompt)
        print('='*25, 'code, driver=', self.driver.get_name(), '='*25, '\n', self.code)

    def review_code(self):
        # Code review method
        review_prompt = f"Review the code below to indicate if there is an error, and only return [NOERROR] if there is no error. The question is [{self.question}]. The code you need to check is [{self.code}]"
        review_results = self.navigator.make_request(review_prompt).replace('\"', '').replace('\\n', '\n')
        self.errors = review_results
        print('='*25, 'review_results, navigator=', self.navigator.get_name(), '='*25, '\n', review_results)

    def fix_errors(self):
        # Method to fix code errors
        fix_prompt = f"Follow the instructions below to fix errors in the code. Your answer only needs to contain the code. The question is [{self.question}]. The code you just generated is [{self.code}]. The review is [{self.errors}]"
        # self.generate_code(fix_prompt)
        self.code = self.driver.make_request(fix_prompt).replace('\"', '').replace('\\n', '\n')
        print('='*25, 'fix_errors, driver=', self.driver.get_name(), '='*25, '\n', self.code)

    def run(self, initial_prompt):
        self.question = initial_prompt
        # Initialize role responsibilities
        self.driver.make_request(self.default_driver_prompt)
        self.navigator.make_request(self.default_navigator_prompt)
        # Main run loop
        self.code = initial_prompt
        self.generate_code(initial_prompt)
        self.review_code()
        while not self.errors.__contains__('NOERROR'):
            self.fix_errors()
            self.review_code()
            self.switch_roles()  # Switch roles after each iteration to ensure participation from both sides
        return self.code

import sys

def generate_one_completion(initial_prompt, id):
    # set_cuda_visible_devices("1,2,3,4")
    sys.stdout = open(f'testgpt4/output-{id}.txt', 'w')

    framework = PairDevFramework(driver = openai_Service(model_name = "gpt-3.5-turbo"),
                                    navigator = openai_Service(model_name = "gpt-3.5-turbo"))


    framework.reset()

    # initial_prompt = '''
    # def histogram(test):
    #     """Given a string representing a space separated lowercase letters, return a dictionary
    #     of the letter with the most repetition and containing the corresponding count.
    #     If several letters have the same occurrence, return all of them.

    #     Example:
    #     histogram('a b c') == {'a': 1, 'b': 1, 'c': 1}
    #     histogram('a b b a') == {'a': 2, 'b': 2}
    #     histogram('a b c a b') == {'a': 2, 'b': 2}
    #     histogram('b b b b a') == {'b': 4}
    #     histogram('') == {}

    #     """
    # '''

    generated_code = framework.run(initial_prompt)
    print('='*25, 'final code', '='*25, '\n', generated_code)

    sys.stdout.close()
    sys.stdout = sys.__stdout__
    return generated_code


In [80]:
# !cd /content/human-eval
!pip install -e /content/human-eval

In [61]:
pip list

Package                          Version               Editable project location
-------------------------------- --------------------- -------------------------
absl-py                          1.4.0
aiohttp                          3.9.5
aiosignal                        1.3.1
alabaster                        0.7.16
albumentations                   1.3.1
altair                           4.2.2
annotated-types                  0.7.0
anyio                            3.7.1
argon2-cffi                      23.1.0
argon2-cffi-bindings             21.2.0
array_record                     0.5.1
arviz                            0.15.1
astropy                          5.3.4
astunparse                       1.6.3
async-timeout                    4.0.3
atpublic                         4.1.0
attrs                            23.2.0
audioread                        3.0.1
autograd                         1.6.2
Babel                            2.15.0
backcall                         0.2.0
beautifulsoup

In [77]:
import human_eval

In [93]:
from human_eval.data import write_jsonl, read_problems

problems = read_problems("/content/human-eval/human_eval/data/HumanEval.jsonl.gz")

num_samples_per_task = 1
samples = [
    dict(task_id=task_id, completion=generate_one_completion(problems[task_id]["prompt"], task_id.replace('HumanEval/','')))
    for task_id in problems
    for _ in range(num_samples_per_task)
]
write_jsonl("samplesgpt4.jsonl", samples)

In [86]:
print("evaluate!")
!evaluate_functional_correctness samplesgpt4.jsonl
print("Done!")